# Notebook 05 - Regressao: previsao numerica de cobertura (nao mais classificacao)

**Troca de abordagem em relacao ao notebook04:** o modelo anterior (regressao
logistica) previa uma pergunta binaria -- "esse municipio vai ter uma queda
ATIPICA (z-score < -2) no ano seguinte?". Esse notebook troca o alvo por uma
pergunta numerica -- "qual vai ser a cobertura GERAL (%) do municipio no ano
seguinte?" -- e usa regressao linear em vez de logistica.

**Alvo = cobertura GERAL do municipio** (media de `cobertura_capada` entre as
5 vacinas, por ibge6 e ano, ignorando NaN) -- a mesma definicao de "perfil
combinado" ja usada no Dia 4 Parte 1 e no z-score de queda atipica do Dia 5.

**O pipeline de features causais (Etapas A/B abaixo) e o MESMO do notebook04,
reaproveitado sem mudanca de logica** -- so o alvo muda de binario pra
numerico. Quem ja leu o notebook04 pode pular direto pra "Etapa C'".

**Fonte de treino: SO 2014-2023 (VacinaBR original).** Os dados reconstruidos
do PNI (2024-2026) nao entram no treino -- so sao usados depois (Etapa G)
pra validar a previsao contra a realidade.

**Aviso de antemao (spoiler do que vem pela frente):** a primeira versao
deste modelo (Etapas D'/E') deu um ajuste ruim -- investigado nas Etapas F/G
abaixo, veio de um motivo diferente do que se esperava (nao foi
multicolinearidade), e foi corrigido com uma mudanca de metodologia (destrend
por tendencia nacional, Etapa H em diante). As duas versoes ficam visiveis
no notebook de proposito, como registro de como o problema foi encontrado e
resolvido -- nao so o resultado final.

In [1]:
import pandas as pd
import numpy as np
import json

pd.set_option("display.max_columns", None)

## Etapa A - cobertura geral estendida (2014-2025) e features causais

A serie de cobertura geral e estendida ate 2025 usando
`data/processed/cobertura_mvp_2014_2026.csv` (2014-2023 = VacinaBR,
2024-2026 = PNI reconstruido, ver CLAUDE.md), que tem os MESMOS valores de
`cobertura_capada` que `serie_temporal_mvp.csv` pros anos 2014-2023
(conferido: diferenca zero). **2026 fica de fora** -- e ano parcial (8
meses), misturar com anos cheios distorceria o alvo.

A funcao `causal_features_grupo` e **identica** ao notebook04 -- janela
expansiva, so olha pra tras. Rodar sobre a serie estendida reproduz
exatamente as linhas de T<=2023 do notebook04 e ganha de graca as linhas de
T=2024 (prontas pra prever 2025).

In [2]:
mvp = pd.read_csv(
    "../data/processed/cobertura_mvp_2014_2026.csv",
    dtype={"ibge6": str, "uf": str, "imunizante": str},
    low_memory=False,
)
mvp = mvp[mvp["ano"] <= 2025]

cobertura_geral = mvp.groupby(["ibge6", "ano"])["cobertura_capada"].mean().reset_index()
fonte_por_ano = mvp.drop_duplicates("ano").set_index("ano")["fonte"].to_dict()
cobertura_geral["fonte"] = cobertura_geral["ano"].map(fonte_por_ano)
cobertura_geral_valid = (
    cobertura_geral.dropna(subset=["cobertura_capada"]).sort_values(["ibge6", "ano"])
)

print(f"Linhas de cobertura_mvp_2014_2026 (<=2025): {len(mvp)}")
print(f"(ibge6, ano) com cobertura geral valida: {len(cobertura_geral_valid)}")
print("\nFonte por ano:")
print(cobertura_geral_valid.groupby("ano")["fonte"].first())

Linhas de cobertura_mvp_2014_2026 (<=2025): 329235
(ibge6, ano) com cobertura geral valida: 66825

Fonte por ano:
ano
2014            vacinabr
2015            vacinabr
2016            vacinabr
2017            vacinabr
2018            vacinabr
2019            vacinabr
2020            vacinabr
2021            vacinabr
2022            vacinabr
2023            vacinabr
2024    pni_reconstruido
2025    pni_reconstruido
Name: fonte, dtype: str


In [3]:
def causal_features_grupo(grupo):
    grupo = grupo.sort_values("ano")
    anos = grupo["ano"].values.astype(float)
    valores = grupo["cobertura_capada"].values
    linhas = []
    for j in range(len(valores)):
        anos_j = anos[: j + 1]
        valores_j = valores[: j + 1]
        n_validos = j + 1
        media = valores_j.mean()
        if n_validos >= 2:
            variab = valores_j.std(ddof=1)
            slope = np.polyfit(anos_j, valores_j, 1)[0]
        else:
            variab = np.nan
            slope = np.nan
        linhas.append({
            "ano": anos[j],
            "n_validos_causal": n_validos,
            "media_causal": media,
            "variabilidade_causal": variab,
            "tendencia_causal": slope,
        })
    return pd.DataFrame(linhas)


causal = (
    cobertura_geral_valid.groupby("ibge6")
    .apply(causal_features_grupo, include_groups=False)
    .reset_index()
    .drop(columns=["level_1"])
)
causal["ano"] = causal["ano"].astype(int)

print(f"Tabela causal (agora ate 2024, pra poder prever 2025): {causal.shape}")
print("anos presentes:", sorted(causal["ano"].unique()))

Tabela causal (agora ate 2024, pra poder prever 2025): (66825, 6)
anos presentes: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


## Etapa B - isolamento e cluster causais por ano T (ate T=2024)

Identico ao notebook04, so que o loop agora vai ate **T=2024**. Validacao de
sempre: em T=2023 os tamanhos de cluster devem bater com
`clusters_municipios_mvp.csv`.

In [4]:
def kmeans_plus_plus_init(X, k, rng):
    n = X.shape[0]
    centros = np.empty((k, X.shape[1]))
    centros[0] = X[rng.integers(n)]
    dist_sq = np.sum((X - centros[0]) ** 2, axis=1)
    for i in range(1, k):
        probs = dist_sq / dist_sq.sum()
        centros[i] = X[rng.choice(n, p=probs)]
        nova_dist_sq = np.sum((X - centros[i]) ** 2, axis=1)
        dist_sq = np.minimum(dist_sq, nova_dist_sq)
    return centros


def kmeans_lloyd(X, k, centros_iniciais, max_iter=300, tol=1e-4):
    centros = centros_iniciais.copy()
    for _ in range(max_iter):
        dists = ((X[:, None, :] - centros[None, :, :]) ** 2).sum(axis=2)
        labels = dists.argmin(axis=1)
        novos_centros = centros.copy()
        for j in range(k):
            membros = X[labels == j]
            if len(membros) > 0:
                novos_centros[j] = membros.mean(axis=0)
        deslocamento = np.sum((novos_centros - centros) ** 2)
        centros = novos_centros
        if deslocamento < tol:
            break
    dists_finais = ((X[:, None, :] - centros[None, :, :]) ** 2).sum(axis=2)
    labels = dists_finais.argmin(axis=1)
    inertia = dists_finais[np.arange(len(X)), labels].sum()
    return labels, centros, inertia


def kmeans_proprio(X, k, n_init=10, random_state=42, max_iter=300):
    rng_principal = np.random.default_rng(random_state)
    melhor_inertia = np.inf
    melhor_labels = melhor_centros = None
    for _ in range(n_init):
        rng = np.random.default_rng(rng_principal.integers(0, 2**32 - 1))
        centros_iniciais = kmeans_plus_plus_init(X, k, rng)
        labels, centros, inertia = kmeans_lloyd(X, k, centros_iniciais, max_iter=max_iter)
        if inertia < melhor_inertia:
            melhor_inertia, melhor_labels, melhor_centros = inertia, labels, centros
    return melhor_labels, melhor_centros, melhor_inertia


def knn_distancias(X, k=5, batch=500):
    n = X.shape[0]
    vizinhos_dist = np.zeros((n, k))
    for inicio in range(0, n, batch):
        fim = min(inicio + batch, n)
        bloco = X[inicio:fim]
        diffs = bloco[:, None, :] - X[None, :, :]
        dists = np.sqrt((diffs ** 2).sum(axis=2))
        for i_local, i_global in enumerate(range(inicio, fim)):
            dists[i_local, i_global] = np.inf
        idx = np.argsort(dists, axis=1)[:, :k]
        for i_local, i_global in enumerate(range(inicio, fim)):
            vizinhos_dist[i_global] = dists[i_local, idx[i_local]]
    return vizinhos_dist

In [5]:
MIN_ANOS_CAUSAL = 3
COLUNAS_3F = ["media_causal", "variabilidade_causal", "tendencia_causal"]

resultados_por_T = {}
for T in range(2014, 2025):
    sub = causal[(causal["ano"] == T) & (causal["n_validos_causal"] >= MIN_ANOS_CAUSAL)].copy()
    if len(sub) < 10:
        continue

    X = sub[COLUNAS_3F].values
    media_c, desvio_c = X.mean(axis=0), X.std(axis=0, ddof=0)
    Xp = (X - media_c) / desvio_c

    labels, _, _ = kmeans_proprio(Xp, k=3, n_init=10, random_state=42)
    sub["cluster_raw"] = labels

    stats = sub.groupby("cluster_raw")[COLUNAS_3F].mean()
    alta = stats["media_causal"].idxmax()
    demais = stats.index.difference([alta])
    recuperacao = stats.loc[demais, "tendencia_causal"].idxmax()
    queda = stats.loc[demais, "tendencia_causal"].idxmin()
    sub["cluster_causal"] = sub["cluster_raw"].map(
        {alta: "alta_estavel", recuperacao: "baixa_recuperacao", queda: "queda_instavel"}
    )

    dist5 = knn_distancias(Xp, k=5, batch=500)
    sub["isolamento_causal"] = dist5.mean(axis=1)

    resultados_por_T[T] = sub[
        ["ibge6", "ano", "media_causal", "variabilidade_causal", "tendencia_causal",
         "cluster_causal", "isolamento_causal"]
    ]
    print(f"T={T}: n={len(sub)} municipios, clusters={sub['cluster_causal'].value_counts().to_dict()}")

causal_completo = pd.concat(resultados_por_T.values(), ignore_index=True)
print(f"\nTabela causal completa (T=2014..2024): {causal_completo.shape}")

T=2016: n=5561 municipios, clusters={'alta_estavel': 3305, 'baixa_recuperacao': 1589, 'queda_instavel': 667}


T=2017: n=5569 municipios, clusters={'alta_estavel': 3243, 'baixa_recuperacao': 1424, 'queda_instavel': 902}


T=2018: n=5570 municipios, clusters={'alta_estavel': 3257, 'baixa_recuperacao': 1202, 'queda_instavel': 1111}


T=2019: n=5568 municipios, clusters={'alta_estavel': 3376, 'queda_instavel': 1382, 'baixa_recuperacao': 810}


T=2020: n=5568 municipios, clusters={'alta_estavel': 2929, 'baixa_recuperacao': 1339, 'queda_instavel': 1300}


T=2021: n=5569 municipios, clusters={'alta_estavel': 2816, 'queda_instavel': 1377, 'baixa_recuperacao': 1376}


T=2022: n=5569 municipios, clusters={'alta_estavel': 2859, 'baixa_recuperacao': 1357, 'queda_instavel': 1353}


T=2023: n=5570 municipios, clusters={'alta_estavel': 2860, 'baixa_recuperacao': 1391, 'queda_instavel': 1319}


T=2024: n=5570 municipios, clusters={'alta_estavel': 2832, 'baixa_recuperacao': 1453, 'queda_instavel': 1285}

Tabela causal completa (T=2014..2024): (50114, 7)


In [6]:
clusters_originais = pd.read_csv(
    "../data/processed/clusters_municipios_mvp.csv", dtype={"ibge6": str}
)
print("Tamanhos originais (Dia 4, 2014-2023 inteiro):")
print(clusters_originais["nome_cluster"].value_counts())
print("\nTamanhos causais em T=2023 (deve bater):")
print(causal_completo[causal_completo["ano"] == 2023]["cluster_causal"].value_counts())

Tamanhos originais (Dia 4, 2014-2023 inteiro):
nome_cluster
Cobertura alta e estavel           2860
Cobertura baixa, em recuperacao    1391
Cobertura em queda e instavel      1319
Name: count, dtype: int64

Tamanhos causais em T=2023 (deve bater):
cluster_causal
alta_estavel         2860
baixa_recuperacao    1391
queda_instavel       1319
Name: count, dtype: int64


## Etapa C' - tabela de treino com alvo NUMERICO (nao mais binario)

`y = cobertura_geral REAL em T+1`, buscado direto na tabela de cobertura
estendida (ja vem certo, seja VacinaBR ou PNI reconstruido). **Treino SO com
T=2014..2022, onde T+1 <= 2023 (fonte VacinaBR).**

Aproveitamos pra ja calcular `cobertura_nacional_t1` (media nacional do ano
T+1) em cada linha -- vai ser usada mais a frente, na correcao de destrend.

In [7]:
alvo = cobertura_geral_valid[["ibge6", "ano", "cobertura_capada", "fonte"]].rename(
    columns={"ano": "ano_t1", "cobertura_capada": "cobertura_t1_real", "fonte": "fonte_t1"}
)

causal_completo["ano_t1"] = causal_completo["ano"] + 1
df = causal_completo.merge(alvo, on=["ibge6", "ano_t1"], how="inner")

nacional_por_ano = cobertura_geral_valid.groupby("ano")["cobertura_capada"].mean()
df["cobertura_nacional_t1"] = df["ano_t1"].map(nacional_por_ano)
df["cobertura_nacional_t"] = df["ano"].map(nacional_por_ano)

print(f"Linhas com causal (T) + alvo (T+1) disponiveis: {len(df)}")
print("\nContagem por ano T e fonte do alvo (T+1):")
print(df.groupby(["ano", "fonte_t1"]).size().unstack(fill_value=0))

print("\nCobertura geral media NACIONAL por ano (contexto pra Etapa F):")
print(nacional_por_ano.round(2))

Linhas com causal (T) + alvo (T+1) disponiveis: 50109

Contagem por ano T e fonte do alvo (T+1):


fonte_t1  pni_reconstruido  vacinabr
ano                                 
2016                     0      5561
2017                     0      5569
2018                     0      5568
2019                     0      5567
2020                     0      5567
2021                     0      5568
2022                     0      5569
2023                  5570         0
2024                  5570         0

Cobertura geral media NACIONAL por ano (contexto pra Etapa F):
ano
2014    84.49
2015    83.26
2016    79.28
2017    81.75
2018    81.09
2019    77.79
2020    73.18
2021    69.91
2022    80.51
2023    88.18
2024    92.10
2025    88.80
Name: cobertura_capada, dtype: float64


In [8]:
df["cluster_baixa_recuperacao"] = (df["cluster_causal"] == "baixa_recuperacao").astype(float)
df["cluster_queda_instavel"] = (df["cluster_causal"] == "queda_instavel").astype(float)
df_treino_pool = df[(df["ano"] <= 2022) & (df["fonte_t1"] == "vacinabr")].copy()

FEATURES_CONTINUAS = ["media_causal", "variabilidade_causal", "tendencia_causal", "isolamento_causal"]
FEATURES_DUMMIES = ["cluster_baixa_recuperacao", "cluster_queda_instavel"]
FEATURES = FEATURES_CONTINUAS + FEATURES_DUMMIES

TREINO_T = list(range(2016, 2021))   # preve 2017-2021
TESTE_T = [2021, 2022]               # preve 2022-2023

treino = df_treino_pool[df_treino_pool["ano"].isin(TREINO_T)].copy()
teste = df_treino_pool[df_treino_pool["ano"].isin(TESTE_T)].copy()

print(f"Treino: {len(treino)} linhas | alvo medio={treino['cobertura_t1_real'].mean():.1f}% "
      f"desvio={treino['cobertura_t1_real'].std():.1f}")
print(f"Teste:  {len(teste)} linhas | alvo medio={teste['cobertura_t1_real'].mean():.1f}% "
      f"desvio={teste['cobertura_t1_real'].std():.1f}")

Treino: 27832 linhas | alvo medio=76.7% desvio=17.0
Teste:  11137 linhas | alvo medio=84.3% desvio=14.3


## Etapa D' - primeira tentativa: regressao direta na cobertura bruta

Equacao normal via `numpy.linalg.lstsq` (estavel mesmo com features
correlacionadas, ao contrario de inverter a matriz na mao). Calculamos o
numero de condicao da matriz de features ANTES de qualquer outra coisa,
porque a suspeita inicial era colinearidade entre os dummies de cluster e as
3 features continuas de onde eles vieram.

In [9]:
media_treino = treino[FEATURES_CONTINUAS].mean().values
desvio_treino = treino[FEATURES_CONTINUAS].std(ddof=0).values

def monta_X(sub, media, desvio):
    cont_padr = (sub[FEATURES_CONTINUAS].values - media) / desvio
    return np.hstack([cont_padr, sub[FEATURES_DUMMIES].values])

def treinar_regressao(X, y):
    n = X.shape[0]
    Xb = np.hstack([np.ones((n, 1)), X])
    beta, residuos, rank, sv = np.linalg.lstsq(Xb, y, rcond=None)
    numero_condicao = sv.max() / sv.min()
    return beta, numero_condicao, rank, Xb.shape[1]

def prever(X, beta):
    Xb = np.hstack([np.ones((X.shape[0], 1)), X])
    return Xb @ beta

def metricas(y_real, y_previsto, rotulo):
    erro = y_previsto - y_real
    erro_abs = np.abs(erro)
    mae = erro_abs.mean()
    mediana_erro = np.median(erro_abs)
    correl = np.corrcoef(y_real, y_previsto)[0, 1]
    ss_res = (erro ** 2).sum()
    ss_tot = ((y_real - y_real.mean()) ** 2).sum()
    r2 = 1 - ss_res / ss_tot
    print(f"{rotulo:28s} MAE={mae:5.2f} p.p. | mediana={mediana_erro:5.2f} p.p. | "
          f"correlacao={correl:+.3f} | R2={r2:+.3f}")
    return {"mae": mae, "mediana": mediana_erro, "correlacao": correl, "r2": r2}

X_treino = monta_X(treino, media_treino, desvio_treino)
y_treino = treino["cobertura_t1_real"].values.astype(float)
X_teste = monta_X(teste, media_treino, desvio_treino)
y_teste = teste["cobertura_t1_real"].values.astype(float)

beta_naive, cond_naive, rank_naive, n_colunas = treinar_regressao(X_treino, y_treino)

nomes_coef = ["intercepto"] + FEATURES
print("Coeficientes (modelo direto, sem destrend):")
for nome, valor in zip(nomes_coef, beta_naive):
    print(f"  {nome:30s} {valor:+.4f}")
print(f"\nPosto da matriz: {rank_naive} de {n_colunas} colunas | Numero de condicao: {cond_naive:.1f}")
print("  (regra pratica: <30 = ok, 30-100 = moderada, >100 = severa -- aqui esta OK,")
print("   entao a colinearidade com os dummies de cluster NAO e o problema)")

Coeficientes (modelo direto, sem destrend):
  intercepto                     +77.7529
  media_causal                   +9.0878
  variabilidade_causal           +0.3406
  tendencia_causal               +1.3784
  isolamento_causal              +1.2111
  cluster_baixa_recuperacao      +0.8941
  cluster_queda_instavel         -6.2726

Posto da matriz: 7 de 7 colunas | Numero de condicao: 8.0
  (regra pratica: <30 = ok, 30-100 = moderada, >100 = severa -- aqui esta OK,
   entao a colinearidade com os dummies de cluster NAO e o problema)


In [10]:
p_treino_naive = prever(X_treino, beta_naive)
p_teste_naive = prever(X_teste, beta_naive)

print("--- Avaliacao do modelo DIRETO (sem destrend) ---")
metricas(y_treino, p_treino_naive, "treino (2017-2021)")
m_teste_naive = metricas(y_teste, p_teste_naive, "teste (2022-2023)")

--- Avaliacao do modelo DIRETO (sem destrend) ---
treino (2017-2021)           MAE=10.60 p.p. | mediana= 8.42 p.p. | correlacao=+0.568 | R2=+0.322
teste (2022-2023)            MAE=13.90 p.p. | mediana=12.75 p.p. | correlacao=+0.387 | R2=-0.351


**Resultado:** número de condição baixo (colinearidade não é o problema), mas
o R² no teste saiu **negativo** -- pior que prever a média. Isso é o "ajuste
ruim" mencionado de antemão. A causa, investigada na próxima célula, não é
overfitting por features correlacionadas -- é a cobertura NACIONAL ter uma
tendência muito forte de ano pra ano (queda até 2021, recuperação depois)
que as features (todas relativas ao próprio município) não capturam.

In [11]:
print("Cobertura geral media NACIONAL por ano:")
print(nacional_por_ano.round(2).to_string())
print()
print(f"Alvo medio no TREINO (T+1 = 2017-2021): {y_treino.mean():.1f}%")
print(f"Alvo medio no TESTE  (T+1 = 2022-2023): {y_teste.mean():.1f}%")
print(f"--> o pais inteiro subiu {y_teste.mean() - y_treino.mean():+.1f} p.p. entre as duas janelas.")
print(f"    O modelo aprendeu o padrao da fase de QUEDA (2016-2021) e foi aplicado na fase")
print(f"    de RECUPERACAO (2022-2023) -- por isso subestima quase todo mundo.")

Cobertura geral media NACIONAL por ano:
ano
2014    84.49
2015    83.26
2016    79.28
2017    81.75
2018    81.09
2019    77.79
2020    73.18
2021    69.91
2022    80.51
2023    88.18
2024    92.10
2025    88.80

Alvo medio no TREINO (T+1 = 2017-2021): 76.7%
Alvo medio no TESTE  (T+1 = 2022-2023): 84.3%
--> o pais inteiro subiu +7.6 p.p. entre as duas janelas.
    O modelo aprendeu o padrao da fase de QUEDA (2016-2021) e foi aplicado na fase
    de RECUPERACAO (2022-2023) -- por isso subestima quase todo mundo.


## Etapa E' - correção: destrend pela tendência nacional

**Conceito (mesma ideia do z-score por ano do Dia 5, aplicada aqui):** em vez
de prever a cobertura bruta, o modelo prevê o **desvio** em relação à média
nacional daquele ano — `desvio_t1 = cobertura_t1_real - cobertura_nacional_t1`.
Isso isola o sinal *relativo* de cada município (está indo melhor ou pior
que a média do país?) do choque nacional (pandemia, recuperação pós-pandemia
etc.), que nenhuma das features municipais capturava.

**Para reconstruir uma previsão absoluta sem trapacear** (sem usar o valor
real futuro, que na aplicação de verdade ainda não existe): a tendência
nacional esperada para T+1 é estimada por **extrapolação linear simples** —
`nacional_T + (nacional_T - nacional_T-1)`, ou seja, "o país continua
mudando no mesmo ritmo do último ano observado". É uma estimativa ingênua de
propósito (não um forecast nacional sofisticado, que seria um projeto à
parte) — mas já é o suficiente pra não repetir o erro de ignorar a
tendência por completo.

`previsao_absoluta = nacional_T1_estimado + desvio_previsto`

In [12]:
df["desvio_t1"] = df["cobertura_t1_real"] - df["cobertura_nacional_t1"]
df_treino_pool = df[(df["ano"] <= 2022) & (df["fonte_t1"] == "vacinabr")].copy()
treino = df_treino_pool[df_treino_pool["ano"].isin(TREINO_T)].copy()
teste = df_treino_pool[df_treino_pool["ano"].isin(TESTE_T)].copy()

y_treino_desvio = treino["desvio_t1"].values.astype(float)
y_teste_desvio = teste["desvio_t1"].values.astype(float)

beta_dtr, cond_dtr, rank_dtr, _ = treinar_regressao(X_treino, y_treino_desvio)
print("Coeficientes (modelo de DESVIO, pos-destrend):")
for nome, valor in zip(nomes_coef, beta_dtr):
    print(f"  {nome:30s} {valor:+.4f}")
print(f"\nNumero de condicao: {cond_dtr:.1f} (mesma matriz X de antes, so muda o alvo -- devia ser igual)")

# extrapolacao linear da tendencia nacional, SEM usar o valor real futuro
def nacional_estimado(ano_t1, serie_nacional):
    ano_t = ano_t1 - 1
    ano_t_menos_1 = ano_t1 - 2
    if ano_t not in serie_nacional.index or ano_t_menos_1 not in serie_nacional.index:
        return np.nan
    delta = serie_nacional[ano_t] - serie_nacional[ano_t_menos_1]
    return serie_nacional[ano_t] + delta

def prever_absoluto(sub, beta, media, desvio):
    X = monta_X(sub, media, desvio)
    desvio_previsto = prever(X, beta)
    nac_est = sub["ano_t1"].map(lambda a: nacional_estimado(a, nacional_por_ano)).values
    return nac_est + desvio_previsto

p_treino_dtr = prever_absoluto(treino, beta_dtr, media_treino, desvio_treino)
p_teste_dtr = prever_absoluto(teste, beta_dtr, media_treino, desvio_treino)

print("\n--- Avaliacao do modelo COM destrend (previsao absoluta reconstruida) ---")
metricas(treino["cobertura_t1_real"].values, p_treino_dtr, "treino (2017-2021)")
m_teste_dtr = metricas(teste["cobertura_t1_real"].values, p_teste_dtr, "teste (2022-2023)")

print("\n--- comparacao lado a lado (teste 2022-2023) ---")
print(f"  SEM destrend: MAE={m_teste_naive['mae']:.2f}  R2={m_teste_naive['r2']:+.3f}")
print(f"  COM destrend: MAE={m_teste_dtr['mae']:.2f}  R2={m_teste_dtr['r2']:+.3f}")

Coeficientes (modelo de DESVIO, pos-destrend):


  intercepto                     +1.4439
  media_causal                   +8.4566
  variabilidade_causal           +1.0944
  tendencia_causal               +2.3437
  isolamento_causal              +0.9679
  cluster_baixa_recuperacao      -1.4071
  cluster_queda_instavel         -5.7938

Numero de condicao: 8.0 (mesma matriz X de antes, so muda o alvo -- devia ser igual)



--- Avaliacao do modelo COM destrend (previsao absoluta reconstruida) ---
treino (2017-2021)           MAE=10.68 p.p. | mediana= 8.58 p.p. | correlacao=+0.578 | R2=+0.328
teste (2022-2023)            MAE=13.83 p.p. | mediana=12.06 p.p. | correlacao=+0.432 | R2=-0.440

--- comparacao lado a lado (teste 2022-2023) ---
  SEM destrend: MAE=13.90  R2=-0.351
  COM destrend: MAE=13.83  R2=-0.440


## Etapa F - modelo final (refit com todo o histórico 2016-2022, já com destrend)

A partir daqui só o modelo com destrend é usado (é o que vai pro artefato
final). O modelo sem destrend fica registrado acima só como parte da
investigação.

In [13]:
treino_final = df_treino_pool[df_treino_pool["ano"].isin(range(2016, 2023))].copy()

media_final = treino_final[FEATURES_CONTINUAS].mean().values
desvio_final = treino_final[FEATURES_CONTINUAS].std(ddof=0).values

X_final = monta_X(treino_final, media_final, desvio_final)
y_final_desvio = treino_final["desvio_t1"].values.astype(float)

beta_final, cond_final, rank_final, _ = treinar_regressao(X_final, y_final_desvio)
print(f"Modelo final (destrend) treinado com {len(treino_final)} linhas (T=2016..2022)")
print(f"Numero de condicao: {cond_final:.1f}")
for nome, valor in zip(nomes_coef, beta_final):
    print(f"  {nome:30s} {valor:+.4f}")

Modelo final (destrend) treinado com 38969 linhas (T=2016..2022)
Numero de condicao: 7.8
  intercepto                     +1.0385
  media_causal                   +7.5094
  variabilidade_causal           +1.2952
  tendencia_causal               +2.4328
  isolamento_causal              +0.3675
  cluster_baixa_recuperacao      -0.1618
  cluster_queda_instavel         -4.7940


## Etapa G - VALIDAÇÃO CONTRA DADO REAL: 2024 e depois 2025 (com destrend)

Mesmo protocolo de não-vazamento do plano original: previsão de 2024 usa
features em T=2023 (só enxerga até 2023) + `beta_final` (treinado só com
2014-2023). Previsão de 2025 usa features em T=2024 (já inclui o 2024 real
na média/variabilidade/tendência/isolamento/cluster), mas com o **mesmo**
`beta_final` — nada é retreinado com 2024 ou 2025.

A extrapolação da tendência nacional para 2024 usa só 2022 e 2023 (números
já conhecidos antes de 2024 existir); para 2025, usa 2023 e 2024 (2024 real,
já conhecido quando chega a hora de prever 2025) — em nenhum dos dois casos
o valor real do próprio ano que está sendo previsto é usado.

**Lembrete importante pra interpretar os números abaixo:** 2024 e 2025 não
são "verdade absoluta" — são reconstruções com incerteza própria (viés
residual de ~1-3% mesmo após a dedup em BCG/Hepatite B, e uma queda de
Pentavalente/Poliomielite em 2025 ainda não totalmente explicada, ver
CLAUDE.md). Parte do erro abaixo pode ser ruído do dado real reconstruído,
não erro do modelo.

In [14]:
df_2024 = df[(df["ano"] == 2023) & (df["fonte_t1"] == "pni_reconstruido")].copy()
p_2024 = prever_absoluto(df_2024, beta_final, media_final, desvio_final)
df_2024["previsto"] = p_2024

print(f"Previsao de 2024 (a partir de T=2023), COM destrend: {len(df_2024)} municipios")
m_2024 = metricas(df_2024["cobertura_t1_real"].values, df_2024["previsto"].values, "2024 real vs previsto")

nac_est_2024 = nacional_estimado(2024, nacional_por_ano)
print(f"\nTendencia nacional extrapolada pra 2024: {nac_est_2024:.1f}% (real foi {nacional_por_ano[2024]:.1f}%)")

Previsao de 2024 (a partir de T=2023), COM destrend: 5570 municipios
2024 real vs previsto        MAE= 7.45 p.p. | mediana= 5.80 p.p. | correlacao=+0.326 | R2=-0.262

Tendencia nacional extrapolada pra 2024: 95.8% (real foi 92.1%)


In [15]:
df_2025 = df[(df["ano"] == 2024) & (df["fonte_t1"] == "pni_reconstruido")].copy()
p_2025 = prever_absoluto(df_2025, beta_final, media_final, desvio_final)
df_2025["previsto"] = p_2025

print(f"Previsao de 2025 (a partir de T=2024, que ja inclui o 2024 real), COM destrend: {len(df_2025)} municipios")
m_2025 = metricas(df_2025["cobertura_t1_real"].values, df_2025["previsto"].values, "2025 real vs previsto")

nac_est_2025 = nacional_estimado(2025, nacional_por_ano)
print(f"\nTendencia nacional extrapolada pra 2025: {nac_est_2025:.1f}% (real foi {nacional_por_ano[2025]:.1f}%)")

print("\n--- RESUMO: SEM destrend vs COM destrend ---")
print(f"{'validacao':28s} {'MAE sem':>10s} {'MAE com':>10s} {'R2 sem':>9s} {'R2 com':>9s}")
print(f"{'2024 (de T=2023)':28s} {13.89:>10.2f} {m_2024['mae']:>10.2f} {-2.040:>9.3f} {m_2024['r2']:>9.3f}")
print(f"{'2025 (de T=2024)':28s} {10.18:>10.2f} {m_2025['mae']:>10.2f} {-0.612:>9.3f} {m_2025['r2']:>9.3f}")

Previsao de 2025 (a partir de T=2024, que ja inclui o 2024 real), COM destrend: 5570 municipios
2025 real vs previsto        MAE= 9.71 p.p. | mediana= 8.04 p.p. | correlacao=+0.402 | R2=-0.719

Tendencia nacional extrapolada pra 2025: 96.0% (real foi 88.8%)

--- RESUMO: SEM destrend vs COM destrend ---
validacao                       MAE sem    MAE com    R2 sem    R2 com
2024 (de T=2023)                  13.89       7.45    -2.040    -0.262
2025 (de T=2024)                  10.18       9.71    -0.612    -0.719


## Etapa H - derivando "risco" da previsão corrigida (sem decidir o corte sozinho)

`queda_prevista = cobertura_T_real - cobertura_T+1_prevista` (positivo =
modelo espera piora). Usamos a previsão mais recente — **T=2024 (real) ->
2025 (previsto)** — com o modelo corrigido.

In [16]:
base_2024 = cobertura_geral_valid[cobertura_geral_valid["ano"] == 2024][["ibge6", "cobertura_capada"]]
base_2024 = base_2024.rename(columns={"cobertura_capada": "cobertura_2024_real"})

risco = df_2025.merge(base_2024, on="ibge6", how="left")
risco["cobertura_2025_prevista"] = risco["previsto"]
risco["queda_prevista"] = risco["cobertura_2024_real"] - risco["cobertura_2025_prevista"]
risco = risco[["ibge6", "cobertura_2024_real", "cobertura_2025_prevista", "queda_prevista"]].dropna()

nomes = pd.read_csv("../data/raw/municipios_nomes.csv", dtype={"ibge6": str})
risco = risco.merge(nomes, on="ibge6", how="left").sort_values("queda_prevista", ascending=False)

print(f"Municipios com risco calculado: {len(risco)}")
print(f"Com queda prevista (>0): {(risco['queda_prevista'] > 0).sum()} "
      f"({(risco['queda_prevista'] > 0).mean() * 100:.1f}%)  "
      f"-- compare com os ~93% da versao sem destrend: deve estar bem mais equilibrado")
print()
print("--- PERCENTIS da queda prevista (p.p.), pra decidir o corte de 'alto risco' ---")
for p in [1, 5, 10, 25, 50, 75, 90, 95, 97, 99]:
    print(f"  p{p:>2d}: {np.percentile(risco['queda_prevista'], p):+6.2f} p.p.")
print()
print("--- 20 maiores quedas previstas ---")
print(risco.head(20)[["ibge6", "nome_municipio", "cobertura_2024_real",
                      "cobertura_2025_prevista", "queda_prevista"]].to_string(index=False))

Municipios com risco calculado: 5570
Com queda prevista (>0): 1382 (24.8%)  -- compare com os ~93% da versao sem destrend: deve estar bem mais equilibrado

--- PERCENTIS da queda prevista (p.p.), pra decidir o corte de 'alto risco' ---
  p 1: -31.85 p.p.
  p 5: -19.59 p.p.
  p10: -14.93 p.p.
  p25:  -9.21 p.p.
  p50:  -4.71 p.p.
  p75:  -0.05 p.p.
  p90:  +3.89 p.p.
  p95:  +5.86 p.p.
  p97:  +7.00 p.p.
  p99:  +8.98 p.p.

--- 20 maiores quedas previstas ---
 ibge6        nome_municipio  cobertura_2024_real  cobertura_2025_prevista  queda_prevista
221062      Sebastião Barros            99.444444                84.029244       15.415201
160025               Itaubal            98.125000                82.854158       15.270842
220530             Jerumenha            97.000000                81.879352       15.120648
220480     Ipiranga do Piauí           100.000000                84.940968       15.059032
220465           Ilha Grande            98.906250                84.132556       1

## Etapa I - salvar o artefato (modelo de regressão com destrend)

`notebook04_risco.ipynb` e `modelo_risco.json` (classificação) ficam
**intocados**, registro histórico da primeira abordagem. O artefato inclui a
série nacional por ano (necessária pra reconstruir a extrapolação na hora de
aplicar) e as métricas de **ambas** as versões (com e sem destrend), pra
documentar por que a correção foi necessária.

In [17]:
artefato = {
    "metadados": {
        "descricao": (
            "Regressao linear (numpy, lstsq) prevendo o DESVIO da cobertura geral em "
            "relacao a tendencia nacional do ano seguinte; previsao absoluta reconstruida "
            "somando a extrapolacao linear da tendencia nacional."
        ),
        "tipo_modelo": "regressao_numerica_destrend",
        "anos_treino_final": list(range(2016, 2023)),
        "n_treino_final": int(len(treino_final)),
        "numero_condicao_treino_final": float(cond_final),
        "metodo_extrapolacao_nacional": "nacional_T + (nacional_T - nacional_T-1)",
        "metricas_SEM_destrend_teste_2022_2023": m_teste_naive,
        "metricas_COM_destrend_teste_2022_2023": m_teste_dtr,
        "validacao_real": {
            "2024_a_partir_de_2023": {**m_2024, "n_municipios": int(len(df_2024))},
            "2025_a_partir_de_2024": {**m_2025, "n_municipios": int(len(df_2025))},
        },
        "aviso": (
            "Validado contra dado reconstruido do PNI (nao VacinaBR oficial), que tem "
            "incerteza propria de ~1-3% mesmo apos dedup. Parte do erro de previsao "
            "pode ser ruido do dado real, nao erro do modelo -- ver CLAUDE.md. A "
            "extrapolacao da tendencia nacional e uma estimativa ingenua (linear simples), "
            "nao um forecast nacional robusto -- erra mais perto de pontos de inflexao."
        ),
    },
    "features_continuas": FEATURES_CONTINUAS,
    "features_dummies": FEATURES_DUMMIES,
    "scaler_mean": media_final.tolist(),
    "scaler_std": desvio_final.tolist(),
    "coeficientes_desvio": {nome: float(valor) for nome, valor in zip(nomes_coef, beta_final)},
    "cobertura_nacional_por_ano": {int(a): float(v) for a, v in nacional_por_ano.items()},
}

with open("../data/processed/modelo_regressao_risco.json", "w", encoding="utf-8") as f:
    json.dump(artefato, f, ensure_ascii=False, indent=2)

print("Artefato salvo em data/processed/modelo_regressao_risco.json")
print(json.dumps(artefato["metadados"], ensure_ascii=False, indent=2))

Artefato salvo em data/processed/modelo_regressao_risco.json
{
  "descricao": "Regressao linear (numpy, lstsq) prevendo o DESVIO da cobertura geral em relacao a tendencia nacional do ano seguinte; previsao absoluta reconstruida somando a extrapolacao linear da tendencia nacional.",
  "tipo_modelo": "regressao_numerica_destrend",
  "anos_treino_final": [
    2016,
    2017,
    2018,
    2019,
    2020,
    2021,
    2022
  ],
  "n_treino_final": 38969,
  "numero_condicao_treino_final": 7.841407268769218,
  "metodo_extrapolacao_nacional": "nacional_T + (nacional_T - nacional_T-1)",
  "metricas_SEM_destrend_teste_2022_2023": {
    "mae": 13.89962340670586,
    "mediana": 12.752105363872658,
    "correlacao": 0.387138489217815,
    "r2": -0.3510865179863818
  },
  "metricas_COM_destrend_teste_2022_2023": {
    "mae": 13.831898616959668,
    "mediana": 12.060342259982335,
    "correlacao": 0.4321946278221029,
    "r2": -0.4398305708903605
  },
  "validacao_real": {
    "2024_a_partir_de_20

In [18]:
# refit final (2016-2022) do modelo SEM destrend, pra comparar 2024/2025 de forma
# justa com os outros metodos nesta mesma rodada (em vez de reusar os numeros
# impressos antes, que vieram de uma versao anterior do notebook)
treino_final_naive = df_treino_pool[df_treino_pool["ano"].isin(range(2016, 2023))].copy()
media_final_naive = treino_final_naive[FEATURES_CONTINUAS].mean().values
desvio_final_naive = treino_final_naive[FEATURES_CONTINUAS].std(ddof=0).values
X_final_naive = monta_X(treino_final_naive, media_final_naive, desvio_final_naive)
y_final_naive = treino_final_naive["cobertura_t1_real"].values.astype(float)
beta_final_naive, cond_final_naive, _, _ = treinar_regressao(X_final_naive, y_final_naive)

df_2024_naive = df[(df["ano"] == 2023) & (df["fonte_t1"] == "pni_reconstruido")].copy()
X_2024_naive = monta_X(df_2024_naive, media_final_naive, desvio_final_naive)
m_2024_naive = metricas(df_2024_naive["cobertura_t1_real"].values, prever(X_2024_naive, beta_final_naive), "2024 (sem destrend)")

df_2025_naive = df[(df["ano"] == 2024) & (df["fonte_t1"] == "pni_reconstruido")].copy()
X_2025_naive = monta_X(df_2025_naive, media_final_naive, desvio_final_naive)
m_2025_naive = metricas(df_2025_naive["cobertura_t1_real"].values, prever(X_2025_naive, beta_final_naive), "2025 (sem destrend)")

2024 (sem destrend)          MAE=13.89 p.p. | mediana=13.40 p.p. | correlacao=+0.348 | R2=-2.040


2025 (sem destrend)          MAE=10.18 p.p. | mediana= 9.71 p.p. | correlacao=+0.436 | R2=-0.612


## Etapa J - última tentativa: modelo de DELTA (mudança) em vez de nível

Mesmas features causais, mesmo numpy/lstsq, mesmo split de treino (2014-2023,
T=2016-2022). A única mudança é o alvo: em vez de prever o **nível** absoluto
`cobertura_t1_real`, prevê a **mudança** `delta_t1 = cobertura_t1_real -
cobertura_t_real` (quanto a cobertura deste município variou de T pra T+1).
A previsão absoluta final é reconstruída como `cobertura_t_real +
delta_previsto` -- mesma ideia da reconstrução do destrend, só que aqui o
que se soma de volta é o valor real do PRÓPRIO município em T (conhecido),
não uma extrapolação da tendência nacional.

Também calculamos aqui, pelos mesmos 3 recortes, o baseline trivial de
**persistência** (`previsto_t1 = cobertura_t_real`, sem modelo nenhum) --
pedido explícito pra comparação final de 4 métodos lado a lado.

In [19]:
# cobertura real em T (pra reconstruir a previsao e pro baseline de persistencia)
cobertura_t_lookup = cobertura_geral_valid.rename(
    columns={"cobertura_capada": "cobertura_t_real"}
)[["ibge6", "ano", "cobertura_t_real"]]
df_delta = df.merge(cobertura_t_lookup, on=["ibge6", "ano"], how="left")
df_delta["delta_t1"] = df_delta["cobertura_t1_real"] - df_delta["cobertura_t_real"]

df_treino_pool_delta = df_delta[(df_delta["ano"] <= 2022) & (df_delta["fonte_t1"] == "vacinabr")].copy()
treino_d = df_treino_pool_delta[df_treino_pool_delta["ano"].isin(TREINO_T)].copy()
teste_d = df_treino_pool_delta[df_treino_pool_delta["ano"].isin(TESTE_T)].copy()

X_treino_d = monta_X(treino_d, media_treino, desvio_treino)
X_teste_d = monta_X(teste_d, media_treino, desvio_treino)
y_treino_delta = treino_d["delta_t1"].values.astype(float)
y_teste_delta = teste_d["delta_t1"].values.astype(float)

beta_delta, cond_delta, rank_delta, _ = treinar_regressao(X_treino_d, y_treino_delta)
print("Coeficientes (modelo DELTA):")
for nome, valor in zip(nomes_coef, beta_delta):
    print(f"  {nome:30s} {valor:+.4f}")
print(f"Numero de condicao: {cond_delta:.1f}")

prev_abs_teste_delta = teste_d["cobertura_t_real"].values + prever(X_teste_d, beta_delta)
m_teste_delta = metricas(teste_d["cobertura_t1_real"].values, prev_abs_teste_delta, "teste (delta)")
m_teste_persist = metricas(teste_d["cobertura_t1_real"].values, teste_d["cobertura_t_real"].values, "teste (persistencia)")

Coeficientes (modelo DELTA):
  intercepto                     -1.9140
  media_causal                   -2.4186
  variabilidade_causal           +0.5126
  tendencia_causal               -5.5272
  isolamento_causal              -0.2406
  cluster_baixa_recuperacao      +0.7455
  cluster_queda_instavel         -0.6819
Numero de condicao: 8.0
teste (delta)                MAE=13.84 p.p. | mediana=11.34 p.p. | correlacao=+0.519 | R2=-0.505
teste (persistencia)         MAE=13.72 p.p. | mediana=10.67 p.p. | correlacao=+0.529 | R2=-0.603


In [20]:
# refit final (2016-2022) do modelo delta, pra aplicar em 2024/2025
treino_final_delta = df_treino_pool_delta[df_treino_pool_delta["ano"].isin(range(2016, 2023))].copy()
media_final_d = treino_final_delta[FEATURES_CONTINUAS].mean().values
desvio_final_d = treino_final_delta[FEATURES_CONTINUAS].std(ddof=0).values
X_final_d = monta_X(treino_final_delta, media_final_d, desvio_final_d)
y_final_delta = treino_final_delta["delta_t1"].values.astype(float)
beta_final_delta, cond_final_delta, _, _ = treinar_regressao(X_final_d, y_final_delta)
print(f"Modelo delta final treinado com {len(treino_final_delta)} linhas | numero de condicao: {cond_final_delta:.1f}")

df_2024_d = df_delta[(df_delta["ano"] == 2023) & (df_delta["fonte_t1"] == "pni_reconstruido")].copy()
X_2024_d = monta_X(df_2024_d, media_final_d, desvio_final_d)
prev_abs_2024_delta = df_2024_d["cobertura_t_real"].values + prever(X_2024_d, beta_final_delta)
m_2024_delta = metricas(df_2024_d["cobertura_t1_real"].values, prev_abs_2024_delta, "2024 (delta)")
m_2024_persist = metricas(df_2024_d["cobertura_t1_real"].values, df_2024_d["cobertura_t_real"].values, "2024 (persistencia)")

df_2025_d = df_delta[(df_delta["ano"] == 2024) & (df_delta["fonte_t1"] == "pni_reconstruido")].copy()
X_2025_d = monta_X(df_2025_d, media_final_d, desvio_final_d)
prev_abs_2025_delta = df_2025_d["cobertura_t_real"].values + prever(X_2025_d, beta_final_delta)
m_2025_delta = metricas(df_2025_d["cobertura_t1_real"].values, prev_abs_2025_delta, "2025 (delta)")
m_2025_persist = metricas(df_2025_d["cobertura_t1_real"].values, df_2025_d["cobertura_t_real"].values, "2025 (persistencia)")

Modelo delta final treinado com 38969 linhas | numero de condicao: 7.8
2024 (delta)                 MAE= 7.22 p.p. | mediana= 5.35 p.p. | correlacao=+0.610 | R2=-0.195
2024 (persistencia)          MAE= 7.14 p.p. | mediana= 4.66 p.p. | correlacao=+0.650 | R2=-0.368
2025 (delta)                 MAE= 6.86 p.p. | mediana= 5.35 p.p. | correlacao=+0.572 | R2=+0.069
2025 (persistencia)          MAE= 6.14 p.p. | mediana= 4.76 p.p. | correlacao=+0.655 | R2=+0.223


## Etapa K - veredito final: os 4 métodos lado a lado

Persistência (sem modelo) vs. nível sem destrend vs. nível com destrend vs.
delta -- MAE e R² nos mesmos 3 recortes, pra decidir se vale continuar
insistindo em regressão linear aqui.

In [21]:
metodos = {
    "Persistencia": {
        "teste": m_teste_persist, "2024": m_2024_persist, "2025": m_2025_persist,
    },
    "Nivel sem destrend": {
        "teste": m_teste_naive, "2024": m_2024_naive, "2025": m_2025_naive,
    },
    "Nivel com destrend": {
        "teste": m_teste_dtr, "2024": m_2024, "2025": m_2025,
    },
    "Delta": {
        "teste": m_teste_delta, "2024": m_2024_delta, "2025": m_2025_delta,
    },
}

print(f"{'metodo':22s} | {'MAE teste':>10s} {'MAE 2024':>9s} {'MAE 2025':>9s} | {'R2 teste':>9s} {'R2 2024':>8s} {'R2 2025':>8s}")
print("-" * 95)
for nome, m in metodos.items():
    print(f"{nome:22s} | {m['teste']['mae']:>10.2f} {m['2024']['mae']:>9.2f} {m['2025']['mae']:>9.2f} | "
          f"{m['teste']['r2']:>+9.3f} {m['2024']['r2']:>+8.3f} {m['2025']['r2']:>+8.3f}")

metodo                 |  MAE teste  MAE 2024  MAE 2025 |  R2 teste  R2 2024  R2 2025
-----------------------------------------------------------------------------------------------
Persistencia           |      13.72      7.14      6.14 |    -0.603   -0.368   +0.223
Nivel sem destrend     |      13.90     13.89     10.18 |    -0.351   -2.040   -0.612
Nivel com destrend     |      13.83      7.45      9.71 |    -0.440   -0.262   -0.719
Delta                  |      13.84      7.22      6.86 |    -0.505   -0.195   +0.069
